# Chains in LangChain (Modernized with LCEL)

## Outline

* LCEL Chains (`prompt | llm`) — replaces `LLMChain`
* Sequential Chains via LCEL piping — replaces `SimpleSequentialChain` / `SequentialChain`
* Router Chain via `RunnableLambda` — replaces `MultiPromptChain` / `LLMRouterChain`

> **Note**: This notebook uses LangChain Expression Language (LCEL) instead of the deprecated `LLMChain`, `SequentialChain`, and `MultiPromptChain` classes.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [ ]:
# Set the model variable
llm_model = os.getenv("OPENAI_MODEL", "gpt-4o")

In [4]:
import os
from pathlib import Path
data_dir =Path(os.getcwd()).parent /"data"

In [5]:
import pandas as pd
df = pd.read_csv(data_dir/'Data.csv')

In [6]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LCEL Chain (replaces LLMChain)

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [8]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [16]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}? Only the name, no explanation."
)

In [17]:
# LCEL: prompt | llm | parser (replaces LLMChain)
chain = prompt | llm | StrOutputParser()

In [18]:
product = "Queen Size Sheet Set"
chain.invoke({"product": product})

'Regal Rest Linens'

## Simple Sequential Chain via LCEL (replaces SimpleSequentialChain)

In [20]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}? Only the name, no explanation."
)

# Chain 1: product → company name (string)
chain_one = first_prompt | llm | StrOutputParser()

In [21]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# Chain 2: company_name → description (string)
chain_two = second_prompt | llm | StrOutputParser()

In [22]:
# Pipe chain_one output as input to chain_two via LCEL
overall_simple_chain = (
    {"company_name": chain_one}
    | chain_two
)

In [23]:
overall_simple_chain.invoke({"product": product})

"Royal Rest Linens offers luxurious, high-quality bedding and linens, ensuring ultimate comfort and elegance for a restful night's sleep."

## Sequential Chain via LCEL + RunnablePassthrough (replaces SequentialChain)

In [24]:
from langchain_core.runnables import RunnablePassthrough

In [25]:
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review → output= English_Review
chain_one = first_prompt | llm | StrOutputParser()

In [26]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review → output= summary
chain_two = second_prompt | llm | StrOutputParser()

In [27]:
# prompt template 3: detect language
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review → output= language
chain_three = third_prompt | llm | StrOutputParser()

In [28]:
# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language → output= followup_message
chain_four = fourth_prompt | llm | StrOutputParser()

In [29]:
# overall_chain: input= Review
# Each step adds its output to the dict, which flows to the next step
# RunnablePassthrough.assign() adds new keys while preserving existing ones
overall_chain = (
    RunnablePassthrough.assign(English_Review=chain_one)
    | RunnablePassthrough.assign(summary=chain_two, language=chain_three)
    | RunnablePassthrough.assign(followup_message=chain_four)
)

In [30]:
review = df.Review[5]
result = overall_chain.invoke({"Review": review})

# Display the outputs
for key in ["English_Review", "summary", "followup_message"]:
    print(f"\n{'='*40}\n{key}:\n{'='*40}\n{result[key]}")


English_Review:
I find the taste mediocre. The foam doesn't hold, it's strange. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?

summary:
The reviewer suspects the coffee pods are either from an old batch or counterfeit due to a mediocre taste and poor foam compared to store-bought versions.

followup_message:
Réponse :

Cher client,

Merci d'avoir partagé vos préoccupations concernant les capsules de café. Nous sommes désolés d'apprendre que la qualité n'a pas répondu à vos attentes. Nous prenons très au sérieux la question de la fraîcheur et de l'authenticité de nos produits. Pourriez-vous nous indiquer plus de détails sur votre achat, comme le numéro de lot ou le lieu d'achat ? Cela nous aidera à enquêter davantage et à résoudre ce problème. Nous vous assurons que notre objectif est de vous fournir des produits de la meilleure qualité possible. Nous vous remercions de votre patience et de votre compréhension. Nous restons à votre disposition

## Router Chain via RunnableLambda (replaces MultiPromptChain / LLMRouterChain)

In [31]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

In [32]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    }
]

In [33]:
from langchain_core.runnables import RunnableLambda

In [34]:
llm = ChatOpenAI(temperature=0, model=llm_model)

In [35]:
# Build a chain for each topic prompt
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    destination_chains[name] = prompt | llm | StrOutputParser()

destinations_str = "\n".join(
    f"{p['name']}: {p['description']}" for p in prompt_infos
)

In [37]:
destinations_str

'physics: Good for answering questions about physics\nmath: Good for answering math questions\nHistory: Good for answering history questions\ncomputer science: Good for answering computer science questions'

In [36]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = default_prompt | llm | StrOutputParser()

In [38]:
def route(info: dict) -> str:
    """Route to the appropriate expert chain based on LLM classification."""
    topic = info["topic"].strip().lower()
    for name in destination_chains:
        if name.lower() in topic:
            print(f"Routing to: {name}")
            return destination_chains[name].invoke({"input": info["input"]})
    print("Routing to: DEFAULT")
    return default_chain.invoke({"input": info["input"]})

In [39]:
router_prompt = ChatPromptTemplate.from_template(
    "Given a question, classify it as exactly one of these categories: "
    "{destinations}\n\n"
    "Respond with ONLY the category name (e.g. 'physics', 'math', 'History', 'computer science'), nothing else.\n"
    "If the question does not fit any category, respond with 'DEFAULT'.\n\n"
    "Question: {input}"
)

# Classification chain
classify_chain = router_prompt | llm | StrOutputParser()

In [40]:
# Full router chain: classify → route → invoke the right expert
chain = (
    RunnablePassthrough.assign(
        topic=lambda x: classify_chain.invoke(
            {"input": x["input"], "destinations": destinations_str}
        )
    )
    | RunnableLambda(route)
)

In [41]:
chain.invoke({"input": "What is black body radiation?"})

Routing to: physics


"Black body radiation refers to the electromagnetic radiation emitted by an idealized object called a black body, which absorbs all incident radiation regardless of frequency or angle of incidence. A black body in thermal equilibrium emits radiation with a characteristic spectrum that depends only on its temperature, not on its shape or composition.\n\nThe spectrum of black body radiation is continuous and has a specific shape described by Planck's law. As the temperature of the black body increases, the peak of the emitted spectrum shifts to shorter wavelengths, a phenomenon known as Wien's displacement law. The total energy emitted across all wavelengths increases with the fourth power of the temperature, as described by the Stefan-Boltzmann law.\n\nBlack body radiation is a fundamental concept in physics because it provides a model for understanding how objects emit and absorb radiation. It played a crucial role in the development of quantum mechanics, as classical physics could not

In [42]:
chain.invoke({"input": "what is 2 + 2"})

Routing to: math


'To solve the problem of finding what \\(2 + 2\\) equals, we can break it down into its basic arithmetic components:\n\n1. **Identify the numbers involved**: The numbers we are adding are 2 and 2.\n2. **Understand the operation**: The operation here is addition, which means we are combining the two numbers to find their total.\n3. **Perform the addition**: Add the two numbers together:\n\n   \\[\n   2 + 2 = 4\n   \\]\n\nTherefore, the answer to the question "what is 2 + 2" is 4.'

In [43]:
chain.invoke({"input": "Why does every cell in our body contain DNA?"})

Routing to: DEFAULT


"Every cell in our body contains DNA because DNA serves as the fundamental blueprint for all biological functions and processes. Here are a few reasons why DNA is present in every cell:\n\n1. **Genetic Information**: DNA carries the genetic instructions necessary for the development, functioning, growth, and reproduction of all living organisms. Each cell needs access to this information to perform its specific functions.\n\n2. **Cellular Function**: Different types of cells perform different functions, and DNA provides the instructions for producing the proteins and molecules required for these functions. For example, muscle cells, nerve cells, and skin cells all have different roles and thus need different sets of proteins.\n\n3. **Replication and Growth**: DNA is essential for cell division, which is necessary for growth and repair. When cells divide, they replicate their DNA so that each new cell has the same genetic information as the parent cell.\n\n4. **Consistency and Stability